# Nhóm câu hỏi về Khả năng Giữ chân và Tần suất (Retention & Frequency)

In [116]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


DATA_PATH = "../Data/Processed/"

In [117]:
df_all = pd.read_csv(DATA_PATH + "result_without_user_no_promotion.csv")

## Tính tỉ lệ NPU

## Tiền xử lý - Tạo các cột `first_trans_date`, `trans_rank`, `days_since_first`

In [118]:
# Chuyển đổi reqDate sang định dạng datetime
# Chuyển về datetime chuẩn trước
df_all['reqDate'] = pd.to_datetime(df_all['reqDate'])

# Chỉ giữ lại phần Ngày (Bỏ Giờ Phút Giây)
df_all['reqDate'] = pd.to_datetime(df_all['reqDate'].dt.date)

# CHÚ Ý: Bạn nên lọc các giao dịch thành công (Tùy thuộc vào data thực tế của transStatus)
df_all = df_all[df_all['transStatus'] == 1] 

# Sắp xếp dữ liệu theo userID và thời gian giao dịch
df_all = df_all.sort_values(by=['userID', 'reqDate'])

df_all['campaignID'] = df_all['campaignID'].astype(str)

# Đánh số thứ tự giao dịch (Rank) cho mỗi user (1 là giao dịch đầu tiên, 2 là thứ 2...)
df_all['trans_rank'] = df_all.groupby('userID')['reqDate'].rank(method='first').astype(int)

# Tách tập NPU (Những giao dịch có rank = 1)
npu_df = df_all[df_all['trans_rank'] == 1].copy()
npu_df = npu_df[['userID', 'reqDate', 'campaignID']]
npu_df.rename(columns={'reqDate': 'first_trans_date', 'campaignID': 'acq_campaignID'}, inplace=True)

# Gộp ngày giao dịch đầu tiên vào bảng gốc để dễ tính toán khoảng cách thời gian
df_merged = df_all.merge(npu_df, on='userID', how='left')

# Tính số ngày tính từ lúc phát sinh giao dịch NPU
df_merged['days_since_first'] = (df_merged['reqDate'] - df_merged['first_trans_date']).dt.days

In [119]:
df_merged.head()

,transID,userID,sof,platform,appID,deviceID,userIP,reqDate,amount,userChargeAmount,...,gender,created_account_date,report_cat,report_sub_cat,days_between,first_request,trans_rank,first_trans_date,acq_campaignID,days_since_first
0,4459108026a94166b60f275b05416fb3,000046fd1fef54ddbc605a4a98594da3,sof3,platform3,12,8ef432b3dbd9fbd9c985b25ea1ee73fa,3a6d697d9b5c24c5f6553d4b83031f63,2022-08-09,100000,90000,...,male,2021-09-25,Access_Service,Access_Service_Prepaid_Product,318,True,1,2022-08-09,7421,0
1,4ad24621c15aa505d3ab95667d231491,000046fd1fef54ddbc605a4a98594da3,sof3,platform3,1215,8ef432b3dbd9fbd9c985b25ea1ee73fa,3a6d697d9b5c24c5f6553d4b83031f63,2022-08-09,1000,10,...,male,2021-09-25,Goods_Transaction,Goods_Transaction_Platform,318,False,2,2022-08-09,7421,0
2,eb6a76cd7049d66c2e1c547421dff99b,000046fd1fef54ddbc605a4a98594da3,sof3,platform3,61,8ef432b3dbd9fbd9c985b25ea1ee73fa,b19149d5b0705ba440b75b965f3816e8,2022-08-21,10000,5000,...,male,2021-09-25,Access_Service,Access_Service_Balance_Increase,330,False,3,2022-08-09,7421,12
3,98b760cd98ddb16ea723df8633399905,000046fd1fef54ddbc605a4a98594da3,sof4,platform3,12,8ef432b3dbd9fbd9c985b25ea1ee73fa,d35d4d121dc20506eabd21ca9e80e850,2022-08-24,20000,20000,...,male,2021-09-25,Access_Service,Access_Service_Prepaid_Product,333,False,4,2022-08-09,7421,15
4,17baee4a7081f1325c102295395c0c00,000046fd1fef54ddbc605a4a98594da3,sof3,platform1,748,unknown,000046fd1fef54ddbc605a4a98594da3,2022-12-05,1000,1000,...,male,2021-09-25,Goods_Transaction,Goods_Transaction_Platform,436,False,5,2022-08-09,7421,118


## **Tỉ lệ sống sót (Retention Date)**

In [120]:
# --- BƯỚC 1: SỬA LOGIC TÍNH FLAG KHÔNG CỘNG DỒN (CÁCH B) ---
# Tách thông tin thời gian thực hiện giao dịch thứ 2 và thứ 3 từ bảng df_merged của bạn
trans_2nd = df_merged[df_merged['trans_rank'] == 2][['userID', 'days_since_first']].rename(columns={'days_since_first': 'days_to_2nd'})
trans_3rd = df_merged[df_merged['trans_rank'] == 3][['userID', 'days_since_first']].rename(columns={'days_since_first': 'days_to_3rd'})

# Gộp vào tập NPU ban đầu
npu_analysis = npu_df.merge(trans_2nd, on='userID', how='left').merge(trans_3rd, on='userID', how='left')

# SỬA Ở ĐÂY: Tính Flag theo từng khoảng cửa sổ thời gian độc lập
# Giao dịch lần 2
npu_analysis['retention_7d_2nd'] = np.where((npu_analysis['days_to_2nd'] >= 0) & (npu_analysis['days_to_2nd'] <= 7), 1, 0)
npu_analysis['retention_30d_2nd'] = np.where((npu_analysis['days_to_2nd'] > 7) & (npu_analysis['days_to_2nd'] <= 30), 1, 0)
npu_analysis['retention_60d_2nd'] = np.where((npu_analysis['days_to_2nd'] > 30) & (npu_analysis['days_to_2nd'] <= 60), 1, 0)

# Giao dịch lần 3 (Nếu bạn muốn giữ lại phân tích cho giao dịch 3)
npu_analysis['retention_7d_3rd'] = np.where((npu_analysis['days_to_3rd'] >= 0) & (npu_analysis['days_to_3rd'] <= 7), 1, 0)
npu_analysis['retention_30d_3rd'] = np.where((npu_analysis['days_to_3rd'] > 7) & (npu_analysis['days_to_3rd'] <= 30), 1, 0)
npu_analysis['retention_60d_3rd'] = np.where((npu_analysis['days_to_3rd'] > 30) & (npu_analysis['days_to_3rd'] <= 60), 1, 0)


retention_by_camp = npu_analysis.groupby('acq_campaignID').agg(
    total_npu=('userID', 'nunique'),
    
    # Các cột tính cho Giao dịch lần 2
    r_7d_2nd=('retention_7d_2nd', 'mean'),
    r_30d_2nd=('retention_30d_2nd', 'mean'),
    r_60d_2nd=('retention_60d_2nd', 'mean'),
    
    # Các cột tính bổ sung cho Giao dịch lần 3
    r_7d_3rd=('retention_7d_3rd', 'mean'),
    r_30d_3rd=('retention_30d_3rd', 'mean'),
    r_60d_3rd=('retention_60d_3rd', 'mean')
).reset_index()

# Danh sách tất cả các cột tỷ lệ cần chuyển sang %
columns_to_pct = [
    'r_7d_2nd', 'r_30d_2nd', 'r_60d_2nd',
    'r_7d_3rd', 'r_30d_3rd', 'r_60d_3rd'
]

# Quy đổi tất cả sang % và làm tròn 2 chữ số
for col in columns_to_pct:
    retention_by_camp[col] = (retention_by_camp[col] * 100).round(2)

# Lọc bỏ các Campaign có ít hơn 30 NPU để đảm bảo ý nghĩa thống kê
retention_table_filtered = retention_by_camp[retention_by_camp['total_npu'] >= 30].copy()

# Xem thử kết quả bảng sau khi đã có thêm giao dịch lần 3
display(retention_table_filtered)

,acq_campaignID,total_npu,r_7d_2nd,r_30d_2nd,r_60d_2nd,r_7d_3rd,r_30d_3rd,r_60d_3rd
4,10071,253,66.80,12.25,4.35,41.90,18.18,7.11
5,10072,84,54.76,20.24,5.95,34.52,21.43,7.14
18,10194,32,34.38,31.25,6.25,15.62,18.75,6.25
19,10195,265,40.38,23.40,4.53,11.70,23.02,9.06
21,10197,32,28.12,15.62,15.62,12.50,12.50,6.25
...,...,...,...,...,...,...,...,...
425,9770,181,43.65,17.68,13.26,23.20,19.34,13.81
426,9771,418,41.87,26.79,11.72,17.70,23.92,17.46
427,9773,129,38.76,24.81,10.08,23.26,18.60,17.83
432,9898,126,54.76,7.94,0.79,45.24,3.97,3.17


In [121]:
retention_table_filtered['convert_2nd_to_3rd_7d'] = np.where(
    retention_table_filtered['r_7d_2nd'] > 0,
    ((retention_table_filtered['r_7d_3rd'] / retention_table_filtered['r_7d_2nd']) * 100).round(2),
    0
)

### 20 Campain có quy mô NPU lớn xem xét lần quay lại thứ 2 như thế nào

In [122]:
top_20_camps = retention_table_filtered.nlargest(20, 'total_npu').sort_values('total_npu', ascending=True)

# Xoay dọc dữ liệu (Melt) để đưa vào Plotly chuẩn phom
top_20_long = top_20_camps.melt(
    id_vars=['acq_campaignID'], 
    value_vars=['r_7d_2nd', 'r_30d_2nd', 'r_60d_2nd'],
    var_name='Timeframe', 
    value_name='Retention_Rate'
)

# Đổi lại tên nhãn hiển thị cho rõ ràng
timeframe_map = {
    'r_7d_2nd': 'Từ 0 đến 7 Ngày',
    'r_30d_2nd': 'Từ 8 đến 30 Ngày',
    'r_60d_2nd': 'Từ 31 đến 60 Ngày'
}
top_20_long['Timeframe'] = top_20_long['Timeframe'].map(timeframe_map)


# --- BƯỚC 4: VẼ BIỂU ĐỒ THANH NGANG NHÓM MỚI ---
fig_cohort_clean = px.bar(
    top_20_long,
    x='Retention_Rate',
    y='acq_campaignID',
    color='Timeframe',
    orientation='h',
    barmode='group',
    text_auto='.1f',
    title="Top 20 Campaign (total_npu cao) - Tỷ lệ NPU quay lại giao dịch lần 2 (Theo từng khoảng thời gian)",
    labels={'acq_campaignID': 'Campaign ID', 'Retention_Rate': 'Tỷ lệ Retention (%)', 'Timeframe': 'Khoảng thời gian'},
    category_orders={"Timeframe": ["Từ 0 đến 7 Ngày", "Từ 8 đến 30 Ngày", "Từ 31 đến 60 Ngày"]},
    height=800
)

fig_cohort_clean.update_layout(
    xaxis_ticksuffix='%', 
    yaxis_type='category',
    legend_title_text='Cửa sổ thời gian'
)
fig_cohort_clean.update_yaxes(dtick=1)

fig_cohort_clean.show()

**Thêm tổng số NPU vào bảng

- Thêm tỉ lệ return trong total

In [123]:
top_20_camps = retention_table_filtered.nlargest(20, 'r_7d_2nd').sort_values('r_7d_2nd', ascending=True)

# Xoay dọc dữ liệu (Melt) để đưa vào Plotly chuẩn phom
top_20_long = top_20_camps.melt(
    id_vars=['acq_campaignID'], 
    value_vars=['r_7d_2nd', 'r_30d_2nd', 'r_60d_2nd'],
    var_name='Timeframe', 
    value_name='Retention_Rate'
)

# Đổi lại tên nhãn hiển thị cho rõ ràng
timeframe_map = {
    'r_7d_2nd': 'Từ 0 đến 7 Ngày',
    'r_30d_2nd': 'Từ 8 đến 30 Ngày',
    'r_60d_2nd': 'Từ 31 đến 60 Ngày'
}
top_20_long['Timeframe'] = top_20_long['Timeframe'].map(timeframe_map)


# --- BƯỚC 4: VẼ BIỂU ĐỒ THANH NGANG NHÓM MỚI ---
fig_cohort_clean = px.bar(
    top_20_long,
    x='Retention_Rate',
    y='acq_campaignID',
    color='Timeframe',
    orientation='h',
    barmode='group',
    text_auto='.1f',
    title="Top 20 Campaign (Tỉ lệ r_7d_2nd cao) - Tỷ lệ NPU quay lại giao dịch lần 2(Theo từng khoảng thời gian)",
    labels={'acq_campaignID': 'Campaign ID', 'Retention_Rate': 'Tỷ lệ Retention (%)', 'Timeframe': 'Khoảng thời gian'},
    category_orders={"Timeframe": ["Từ 0 đến 7 Ngày", "Từ 8 đến 30 Ngày", "Từ 31 đến 60 Ngày"]},
    height=800
)

fig_cohort_clean.update_layout(
    xaxis_ticksuffix='%', 
    yaxis_type='category',
    legend_title_text='Cửa sổ thời gian'
)
fig_cohort_clean.update_yaxes(dtick=1)

fig_cohort_clean.show()

### 1. Phát hiện các "Siêu chiến dịch" (Giao điểm của Quy mô & Chất lượng)

**Điểm đắt giá nhất khi so sánh 2 biểu đồ** là tìm ra những Campaign xuất hiện ở cả hai danh sách. Đó chính là Campaign 0 (Organic) và Campaign 7424.

- Campaign 0 (Lượng Organic chuẩn): Nằm trong top gánh volume lớn nhất hệ thống (biểu đồ 1) và có tỷ lệ quay lại 7 ngày đầu gần như tuyệt đối ~99% (biểu đồ 2). Nhóm này xác lập một "vạch đích chuẩn" về hành vi tự nhiên để các chiến dịch marketing khác hướng tới.

- Campaign 7424 (Ngôi sao thực chiến): Đây là chiến dịch marketing xuất sắc nhất. Ở biểu đồ 1, nó mang lại lượng NPU rất lớn, và ở biểu đồ 2, nó chứng minh chất lượng vượt trội với gần 90% user quay lại ngay trong 1 tuần đầu.

Insight: Chiến dịch này đã tiếp cận chính xác tệp khách hàng mục tiêu và có luồng onboarding/incentive (khuyến mãi) cực kỳ hiệu quả, giữ chân khách hàng sâu mà không bị rụng rơi luồng muộn.

### 2. Nhóm "Bẫy Số Lượng" (Quy mô lớn nhưng Rụng rơi quá nhanh)
Hãy nhìn vào các Campaign đứng đầu về số lượng ở Biểu đồ 1 như 8932, 8386, 10249 nhưng hoàn toàn biến mất ở Biểu đồ 2 (Top 7 ngày tốt nhất).

- Hiện tượng: Ở biểu đồ 1, tỷ lệ quay lại 7 ngày đầu của các chiến dịch này khá thấp (chỉ dao động từ 25% đến 35%).

- Insight: Đây là biểu hiện của việc chạy quảng cáo đại trà hoặc "đốt tiền" để lấy số lượng user đăng ký mới (CPA ngon). Tuy nhiên, tệp người dùng mang về có chất lượng rất thấp. Họ vào app thực hiện giao dịch đầu tiên để nhận ưu đãi rồi nhanh chóng rời bỏ hệ thống.

- Khuyến nghị: Cần tối ưu hoặc cắt giảm ngân sách ở nhóm này vì chúng mang lại giá trị vòng đời (LTV) kém, gây lãng phí chi phí marketing.

### 3. Nhóm "Chất lượng ngách" (Chỉ xuất hiện ở Biểu đồ 7 ngày cao nhất)
Một loạt Campaign ở Biểu đồ 2 như 8323, 8985, 9081, 9952, 9140, 7819 đạt tỷ lệ giữ chân 7 ngày gần như 100% nhưng không hề lọt vào Top Volume của biểu đồ 1.

- Insight: Đây là các chiến dịch "nhỏ nhưng có võ". Quy mô tệp khách hàng mang về không quá lớn (hoặc ngân sách chạy chiến dịch nhỏ), nhưng tệp user lọt vào lại cực kỳ chất lượng hoặc sản phẩm đánh trúng nhu cầu 100%.

- Khuyến nghị: Team Marketing nên cân nhắc mở rộng quy mô (Scale-up), đổ thêm ngân sách vào các chiến dịch này vì chúng đang có tỷ lệ chuyển đổi và kích hoạt user tối ưu nhất hệ thống.

## **Tỉ lệ quay lại lần 3**

In [124]:
# 1. Tính toán cột tỷ lệ chuyển đổi sâu (Habit Formation Rate) từ dữ liệu đã nhân 100
# Dùng np.where để phòng trường hợp có campaign nào đó r_7d_2nd bằng 0 (tránh lỗi chia cho 0)
retention_table_filtered['convert_2nd_to_3rd_7d'] = np.where(
    retention_table_filtered['r_7d_2nd'] > 0,
    ((retention_table_filtered['r_7d_3rd'] / retention_table_filtered['r_7d_2nd']) * 100).round(2),
    0
)

# 2. Lọc lấy Top 20 Campaign có quy mô NPU lớn nhất để phân tích
top_20_conversion = retention_table_filtered.nlargest(20, 'convert_2nd_to_3rd_7d').copy()

# Sắp xếp tăng dần theo tỷ lệ chuyển đổi để khi vẽ thanh dài nhất nằm trên cùng
top_20_conversion = top_20_conversion.sort_values('convert_2nd_to_3rd_7d', ascending=True)

# 3. Vẽ biểu đồ thanh ngang đơn giản, thoáng mắt
fig_conversion = px.bar(
    top_20_conversion,
    x='convert_2nd_to_3rd_7d',
    y='acq_campaignID',
    orientation='h',
    text_auto='.1f',
    title="Tỷ lệ giữ chân sâu: % Người dùng giao dịch lần 2 tiếp tục đi đến lần 3 (Trong 7 ngày đầu)",
    labels={
        'acq_campaignID': 'Campaign ID',
        'convert_2nd_to_3rd_7d': 'Tỷ lệ chuyển đổi Lần 2 -> Lần 3 (%)'
    },
    height=600
)

# 4. Tối ưu giao diện theo phom cũ của bạn
fig_conversion.update_traces(marker_color='mediumpurple') # Đổi sang màu tím cho mới mẻ và phân biệt với các biểu đồ trước
fig_conversion.update_layout(xaxis_ticksuffix='%', yaxis_type='category')
fig_conversion.update_yaxes(dtick=1)

fig_conversion.show()

## **Nhịp độ giao dịch**

In [125]:
time_delta = npu_analysis.dropna(subset=['days_to_2nd'])
avg_time_delta = time_delta.groupby('acq_campaignID')['days_to_2nd'].mean().reset_index()
avg_time_delta.rename(columns={'days_to_2nd': 'avg_days_to_2nd_trans'}, inplace=True)
avg_time_delta['avg_days_to_2nd_trans'] = avg_time_delta['avg_days_to_2nd_trans'].round(1)
avg_time_delta = avg_time_delta.sort_values('avg_days_to_2nd_trans')

analysis_matrix = retention_table_filtered.merge(
    avg_time_delta[['acq_campaignID', 'avg_days_to_2nd_trans']], 
    left_on='acq_campaignID', # hoặc 'acq_campaignID' tùy theo tên cột hiện tại của bạn
    right_on='acq_campaignID', 
    how='inner'
)

### Biểu đồ Heatmap

In [126]:
# 1. Phân nhóm Số ngày trung bình (Trục X cũ)
analysis_matrix['Nhóm Tốc Độ (Time Delta)'] = pd.cut(
    analysis_matrix['avg_days_to_2nd_trans'],
    bins=[-np.inf, 3, 7, np.inf],
    labels=['1. Siêu tốc (<=3 ngày)', '2. Vừa phải (3-7 ngày)', '3. Chậm (>7 ngày)']
)

# 2. Phân nhóm Tỷ lệ lên lần 3 (Trục Y cũ)
analysis_matrix['Nhóm Chất Lượng (Lần 2 -> 3)'] = pd.cut(
    analysis_matrix['convert_2nd_to_3rd_7d'],
    bins=[-np.inf, 40, 70, np.inf],
    labels=['C. Thấp (<40%)', 'B. Trung bình (40-70%)', 'A. Cao (>70%)']
)

# 3. Tạo bảng ma trận chéo (Pivot Table) để đếm số lượng Campaign trong mỗi ô
matrix_summary = pd.crosstab(
    analysis_matrix['Nhóm Chất Lượng (Lần 2 -> 3)'],
    analysis_matrix['Nhóm Tốc Độ (Time Delta)']
)

# 4. Vẽ Heatmap bằng Plotly
fig_easy_matrix = px.imshow(
    matrix_summary,
    text_auto=True, # Hiện số lượng Campaign ngay trên ô
    labels=dict(x="Nhịp độ quay lại lần 2", y="Độ gắn kết lên lần 3", color="Số lượng Camp"),
    x=matrix_summary.columns,
    y=matrix_summary.index,
    color_continuous_scale='Blues',
    title="Ma Trận Sức Khỏe Chiến Dịch: Tốc Độ vs Độ Gắn Kết Sâu"
)

fig_easy_matrix.show()

In [127]:
# Lọc lấy danh sách các Campaign "Ngôi sao" hàng đầu
ngoei_sao_camps = analysis_matrix[
    (analysis_matrix['Nhóm Tốc Độ (Time Delta)'] == '1. Siêu tốc (<=3 ngày)') & 
    (analysis_matrix['Nhóm Chất Lượng (Lần 2 -> 3)'] == 'A. Cao (>70%)')
]

print("Danh sách các Campaign NGÔI SAO (Quay lại nhanh + Gắn kết sâu):")
display(ngoei_sao_camps[['acq_campaignID', 'total_npu', 'avg_days_to_2nd_trans', 'convert_2nd_to_3rd_7d']])

Danh sách các Campaign NGÔI SAO (Quay lại nhanh + Gắn kết sâu):


,acq_campaignID,total_npu,avg_days_to_2nd_trans,convert_2nd_to_3rd_7d
9,10443,92,0.3,96.39
16,10670,109,0.1,89.88
17,10675,153,0.0,80.31
18,6580,50,1.5,97.96
51,8501,31,0.4,93.55
74,8985,66,0.0,89.39
79,9097,232,0.3,99.11
94,9645,89,0.5,89.03
96,9650,193,1.2,95.57
97,9651,166,0.4,84.31


### Hiện tượng "Mồi lửa tức thì" (Kích hoạt đồng thời trong ngày)
Nhìn vào bảng dữ liệu, có rất nhiều chiến dịch sở hữu chỉ số avg_days_to_2nd_trans cực kỳ nhỏ, thậm chí bằng đúng 0.0 ngày (như 9081, 9739, 9952) hoặc 0.1 - 0.2 ngày (như 10670, 7819, 9140).

- Bản chất hành vi: Điều này có nghĩa là khi người dùng thực hiện giao dịch NPU (giao dịch đầu tiên), họ lập tức phát sinh tiếp giao dịch lần 2 ngay trong ngày hôm đó (hoặc chỉ sau vài tiếng).

- Hệ quả giữ chân: Chính vì nhịp độ quay lại lần 2 diễn ra quá nhanh và dồn dập, tâm lý của tệp khách hàng này vẫn đang trong trạng thái "nóng sốt". Do đó, họ dễ dàng bị cuốn theo luồng trải nghiệm để thực hiện tiếp giao dịch lần 3 ngay sau đó, dẫn đến tỷ lệ convert_2nd_to_3rd_7d trên biểu đồ thanh ngang đạt mức gần như tuyệt đối (Từ 81.2% đến 100%).

- Kết luận: Tốc độ kích hoạt ban đầu (Time Delta) tỷ lệ thuận với độ dính sâu của người dùng trong ngắn hạn. Bạn đưa khách hàng quay lại càng nhanh, cơ hội biến họ thành khách hàng quen thuộc (đạt đơn thứ 3) càng cao.

### Sự phân hóa về Quy mô của nhóm "Ngôi sao"
Mặc dù tất cả các chiến dịch xuất hiện trong bảng đều là những chiến dịch xuất sắc trong việc thúc đẩy người dùng đi đến giao dịch lần 3, nhưng quy mô đóng góp (total_npu) của chúng lại có sự chênh lệch rất lớn.

- Nhóm Quy mô Ngách (Niche): Các Campaign như 9081 (30 NPU), 8985 (32 NPU), 9739 (32 NPU), 9097 (34 NPU). Nhóm này mang lại tỷ lệ chuyển đổi hoàn hảo nhưng số lượng khách hàng mang về rất ít. Đây thường là các chiến dịch chạy thử nghiệm (A/B Testing) hoặc nhắm vào một tệp đối tượng cực kỳ đặc thù.

- Nhóm Quy mô Chiến lược (Mass): Các Campaign như 9140 (233 NPU) hay 8935 (214 NPU). Đây mới là những "điểm sáng" thực sự cho doanh nghiệp. Chúng vừa chứng minh được khả năng mở rộng quy mô (mang về lượng khách hàng lớn gấp 6 - 7 lần nhóm ngách), vừa giữ vững được phong độ chất lượng với tỷ lệ lên đơn lần 3 đạt trên 93% - 98% và thời gian quay lại lần 2 chưa tới 1 ngày.

### Đề xuất hành động
- Đối với nhóm Chiến lược (9140, 8935): Đây là công thức thành công chuẩn đã được chứng minh. Cần phân tích sâu xem cấu trúc chương trình khuyến mãi, kênh phân phối, hoặc giao diện onboarding của 2 campaign này là gì để áp dụng rộng rãi (scale-up) cho toàn bộ hệ thống.

- Đối với nhóm Ngách (9097, 9739, 6580): Tỷ lệ chuyển đổi đạt 100% chứng tỏ luồng giữ chân cực tốt. Team Marketing nên xem xét tăng thêm ngân sách quảng cáo cho các chiến dịch này để kiểm tra xem khi tệp khách hàng phình to ra thì chất lượng có bị loãng đi hay không.

## Tần suất giao dịch

In [128]:
user_tx_count = df_merged.groupby(['acq_campaignID', 'userID']).size().reset_index(name='total_transactions')

# 2. Tính Tần suất giao dịch trung bình (Average Frequency) của mỗi Campaign
avg_frequency = user_tx_count.groupby('acq_campaignID')['total_transactions'].mean().reset_index()
avg_frequency.rename(columns={'total_transactions': 'avg_tx_per_npu'}, inplace=True)
avg_frequency['avg_tx_per_npu'] = avg_frequency['avg_tx_per_npu'].round(1)

# 3. Gộp thêm cột total_npu từ bảng cũ sang để làm bộ lọc Threshold
# (Đảm bảo chỉ lấy các campaign thực chiến >= 30 NPU)
final_frequency = avg_frequency.merge(
    retention_table_filtered[['acq_campaignID', 'total_npu']], 
    left_on='acq_campaignID', 
    right_on='acq_campaignID', 
    how='inner'
)

# Sắp xếp tăng dần theo tần suất để vẽ biểu đồ thanh ngang đẹp nhất
final_frequency = final_frequency.sort_values('avg_tx_per_npu', ascending=True)

In [129]:
# Lấy Top 20 Campaign có tần suất giao dịch trung bình cao nhất
top_20_freq = final_frequency.nlargest(20, 'avg_tx_per_npu').sort_values('avg_tx_per_npu', ascending=True)

# Vẽ biểu đồ thanh ngang
fig_freq = px.bar(
    top_20_freq,
    x='avg_tx_per_npu',
    y='acq_campaignID',
    orientation='h',
    text_auto='.1f',
    title="Top 20 Campaign có Tần suất Giao dịch trung bình cao nhất của NPU trong 6 tháng",
    labels={'acq_campaignID': 'Campaign ID', 'avg_tx_per_npu': 'Số giao dịch trung bình / NPU'},
    height=600
)

# Tối ưu giao diện theo tone màu của bạn (Chọn màu xanh lá đậm đại diện cho dòng tiền/tần suất)
fig_freq.update_traces(marker_color='forestgreen')
fig_freq.update_layout(xaxis_ticksuffix=' đơn', yaxis_type='category')
fig_freq.update_yaxes(dtick=1)

fig_freq.show()

### Hệ số tương quan giữua SỐ đơn hàng và số tháng gắn bó

In [130]:

# 1. Tính toán cho mỗi user: Tổng số đơn và khoảng thời gian gắn bó thực tế (tính bằng tháng)
user_loyalty = df_merged.groupby('userID').agg(
    total_orders=('days_since_first', 'count'),          # Tổng số đơn hàng đã mua
    max_days_active=('days_since_first', 'max')          # Số ngày từ đơn đầu đến đơn cuối cùng
).reset_index()

# 2. Quy đổi số ngày active tối đa sang số tháng (Cộng thêm 1 để user mua trong tháng đầu được tính là 1 tháng)
user_loyalty['active_months'] = (user_loyalty['max_days_active'] / 30).astype(int) + 1

# 3. Tính hệ số tương quan Pearson giữa Số đơn và Số tháng active
correlation = user_loyalty['total_orders'].corr(user_loyalty['active_months'])

print(f"Hệ số tương quan giữa Số đơn hàng và Số tháng gắn bó: {correlation:.2f}")

Hệ số tương quan giữa Số đơn hàng và Số tháng gắn bó: 0.29


### Heatmap top 20 campain có tần suất giao dịch cao trong 6 tháng

In [131]:
# df_merged['active_month_index'] = df_merged['days_since_first'] // 30

# # Lọc bỏ các giao dịch vượt quá 6 tháng (180 ngày) nếu có
# df_active_months = df_merged[df_merged['active_month_index'] < 6].copy()

# # 2. Tính số lượng user độc nhất (Active) của từng Campaign theo từng Tháng index
# campaign_monthly_active = df_active_months.groupby(['acq_campaignID', 'active_month_index'])['userID'].nunique().reset_index()

# # 3. Lấy số lượng NPU ban đầu (Tháng 0) để làm gốc tính tỷ lệ %
# npu_base = campaign_monthly_active[campaign_monthly_active['active_month_index'] == 0][['acq_campaignID', 'userID']].rename(columns={'userID': 'npu_total'})

# # Gộp vào bảng monthly active
# cohort_matrix = campaign_monthly_active.merge(npu_base, on='acq_campaignID', how='inner')

# # Tính tỷ lệ % Active duy trì qua từng tháng
# cohort_matrix['active_rate'] = ((cohort_matrix['userID'] / cohort_matrix['npu_total']) * 100).round(1)

# # 4. Chuyển đổi sang dạng bảng Pivot rộng (Wide Format) để vẽ Heatmap
# heatmap_data = cohort_matrix.pivot(index='acq_campaignID', columns='active_month_index', values='active_rate').fillna(0)

# # Đổi tên cột cho sếp dễ đọc
# heatmap_data.columns = [f'Tháng {i}' for i in range(6)]

# # 5. Lọc lấy đúng danh sách Top 20 Campaign từ biểu đồ newplot3 của bạn để đối chiếu trực tiếp
# # (Giả sử top_20_freq['acq_campaignID'] chứa danh sách ID của 20 camp đó)
# top_20_ids = top_20_freq['acq_campaignID'].tolist()
# heatmap_data_top20 = heatmap_data.loc[heatmap_data.index.isin(top_20_ids)]

# # Sắp xếp lại các dòng theo thứ tự của biểu đồ tần suất để dễ đối chiếu song song
# heatmap_data_top20 = heatmap_data_top20.reindex(top_20_ids[::-1])

# fig_monthly_heatmap = px.imshow(
#     heatmap_data_top20,
#     text_auto='.1f', # Hiện số % ngay trên ô màu
#     labels=dict(x="Vòng đời người dùng (Theo Tháng)", y="Campaign ID", color="Tỷ lệ Active (%)"),
#     x=heatmap_data_top20.columns,
#     y=heatmap_data_top20.index,
#     color_continuous_scale='RdYlGn', # Màu từ Đỏ (thấp) -> Vàng (vừa) -> Xanh lá (cao) để thấy độ rụng rơi
#     title="Heatmap Duy Trì Hoạt Động Theo Tháng của Top 20 Campaign có Tần suất cao nhất",
#     height=800,
# )

# fig_monthly_heatmap.update_layout(yaxis_type='category')
# fig_monthly_heatmap.show()

In [132]:
# ======================== CHUẨN BỊ DỮ LIỆU ========================
user_first_trans = df_merged.groupby('userID')['reqDate'].min().reset_index()
user_first_trans['acquisition_month'] = pd.to_datetime(user_first_trans['reqDate']).dt.to_period('M')
df_merged = df_merged.merge(user_first_trans[['userID', 'acquisition_month']], on='userID', how='left')

df_merged['transaction_month'] = pd.to_datetime(df_merged['reqDate']).dt.to_period('M')
df_merged['calendar_month_index'] = (df_merged['transaction_month'] - df_merged['acquisition_month']).apply(lambda x: x.n)

df_active_months = df_merged[
    (df_merged['calendar_month_index'] >= 0) & (df_merged['calendar_month_index'] < 6)
].copy()

# ======================== BƯỚC 1: NPU MỚI TỪNG THÁNG ========================
# Chỉ lấy các dòng là giao dịch đầu tiên của user (calendar_month_index == 0)
# → Đây là tháng Campaign thực sự "mang về" user đó
new_npu_per_month = (
    df_active_months[df_active_months['calendar_month_index'] == 0]
    .groupby(['acq_campaignID', 'transaction_month'])['userID']
    .nunique()
    .reset_index()
    .rename(columns={'userID': 'new_npu'})
    .sort_values(['acq_campaignID', 'transaction_month'])
)

# ======================== BƯỚC 2: NPU CỘNG DỒN TỪNG THÁNG ========================
# cumsum() theo từng campaign → tháng 1: NPU_t1, tháng 2: NPU_t1 + NPU_t2, ...
new_npu_per_month['cumulative_npu'] = (
    new_npu_per_month.groupby('acq_campaignID')['new_npu'].cumsum()
)

# ======================== BƯỚC 3: USER ACTIVE TỪNG THÁNG ========================
campaign_monthly_active = (
    df_active_months
    .groupby(['acq_campaignID', 'transaction_month'])['userID']
    .nunique()
    .reset_index()
    .rename(columns={'userID': 'active_users'})
    .sort_values(['acq_campaignID', 'transaction_month'])
)

# ======================== BƯỚC 4: GHÉP VÀ TÍNH TỶ LỆ ========================
cohort_matrix = campaign_monthly_active.merge(
    new_npu_per_month[['acq_campaignID', 'transaction_month', 'cumulative_npu']],
    on=['acq_campaignID', 'transaction_month'],
    how='left'
)

# Với các tháng không có NPU mới (cumulative_npu bị NaN sau merge),
# forward fill để kế thừa số NPU cộng dồn từ tháng trước
cohort_matrix['cumulative_npu'] = (
    cohort_matrix.groupby('acq_campaignID')['cumulative_npu'].ffill()
)

# Tính tỷ lệ active / cumulative NPU
cohort_matrix['active_rate'] = (
    (cohort_matrix['active_users'] / cohort_matrix['cumulative_npu']) * 100
).round(1)

# Đổi sang string để dùng làm cột
cohort_matrix['transaction_month'] = cohort_matrix['transaction_month'].astype(str)

# ======================== BƯỚC 5: PIVOT & VẼ HEATMAP ========================
heatmap_data = cohort_matrix.pivot(
    index='acq_campaignID',
    columns='transaction_month',
    values='active_rate'
)
heatmap_data = heatmap_data.reindex(sorted(heatmap_data.columns), axis=1)

top_20_ids = top_20_freq['acq_campaignID'].tolist()
heatmap_data_top20 = heatmap_data.loc[heatmap_data.index.isin(top_20_ids)]
heatmap_data_top20 = heatmap_data_top20.reindex(top_20_ids[::-1])

# ✅ Kiểm tra: Tháng đầu của mỗi campaign phải = 100.0
print("Kiểm tra - Max value (phải là 100.0):", heatmap_data_top20.max().max())
print("Kiểm tra - Có giá trị > 100 không:", (heatmap_data_top20 > 100).any().any())

fig_monthly_heatmap = px.imshow(
    heatmap_data_top20,
    text_auto='.1f',
    labels=dict(
        x="Tháng Lịch",
        y="Campaign ID",
        color="Tỷ lệ Active (%)"
    ),
    x=heatmap_data_top20.columns.tolist(),
    y=heatmap_data_top20.index.astype(str).tolist(),
    color_continuous_scale='RdYlGn',
    zmin=0, zmax=100,
    title="Heatmap Duy Trì Hoạt Động (Mẫu số: Cumulative NPU) - Top 20 Campaign",
    height=800,
)

fig_monthly_heatmap.update_layout(yaxis_type='category')
fig_monthly_heatmap.show()


Kiểm tra - Max value (phải là 100.0): 100.0
Kiểm tra - Có giá trị > 100 không: False


Biểu đồ Heatmap Duy trì hoạt động đã chứng minh con số tương quan $0.29$ là hoàn toàn thực tế. Doanh nghiệp không thể đánh giá sức khỏe chiến dịch dựa trên tổng số đơn hàng. > Có những chiến dịch mang lại số đơn rất cao nhưng thực chất là 'bẫy số lượng' gãy luồng ngay sau tháng đầu tiên (9097, 10433). Ngược lại, muốn tìm kiếm sự tăng trưởng bền vững, doanh nghiệp phải đầu tư vào những chiến dịch có mô hình giữ chân vững chắc như 7427, 7050, và 7880 — nơi người dùng thực sự gắn bó và chi tiêu phân bổ đều qua từng tháng

-> Vẽ lại heatmap lọc thoe thời gian

## NHẬN XÉT CHUNG

### 1. Sự sụp đổ của "Chỉ số ảo" (Vanity Metrics)
Dữ liệu đã chứng minh rằng Tổng số lượng User mới (NPU) hay Tổng số đơn hàng là những cái bẫy chết người nếu đứng độc lập.

- Hệ số tương quan chỉ đạt mức 0.29 đã vạch trần sự thật: Việc một người dùng cày 15-20 đơn hàng không hề đảm bảo họ sẽ gắn bó với ứng dụng.

- Các chiến dịch như 8932 (đứng đầu về lượng NPU) hay 9097, 10433 (đứng đầu về số đơn) thực chất chỉ là những chiến dịch "đốt tiền" tạo ra khách hàng ăn xổi. Họ đến vì khuyến mãi và rời đi ngay khi hết voucher, để lại một biểu đồ Heatmap trắng xóa ở các tháng tiếp theo và mang lại giá trị trọn đời (LTV) bằng 0.

### 2. Độ giữ chân (Retention) là thước đo đo lường Giá trị thực (True Value)
Thành công của Marketing không nằm ở việc kéo khách hàng đến cửa, mà nằm ở việc giữ họ lại trong nhà. Chất lượng giữ chân được chia làm 2 giai đoạn quyết định:

- Ngắn hạn (Hiệu ứng mồi lửa): Kích hoạt người dùng giao dịch lần 2 cực nhanh (Time Delta tiệm cận 0) để tạo đà ép tỷ lệ đi tiếp lần 3 đạt trên 90%.

- Dài hạn (Sức bền thói quen): Chiến dịch thành công thực sự là chiến dịch tạo ra được các dòng xanh mướt trên Heatmap (duy trì 40% - 60% user hoạt động đều đặn qua 6 tháng) như các chiến dịch 7880, 7427, 7050. Đây mới là cỗ máy in tiền bền vững của doanh nghiệp.

### Chất lượng người dùng

- Độ nhạy và Tốc độ phản hồi (Đo bằng avg_days_to_2nd_trans)

- Sức bền và Khả năng tạo thói quen (Đo bằng convert_2nd_to_3rd_7d)

- Năng lực chống chọi với Churn (Tỷ lệ rụng rơi thấp): Khi họ đã lọt vào phễu giữ chân, tỷ lệ rụng rơi giữa lần 2 và lần 3 là cực kỳ thấp (chỉ mất đi từ 0% đến tối đa 25%). Đây chính là định nghĩa chuẩn về "Tệp người dùng giá trị cao - High-value Users".


$$\text{CQS} = \text{Tốc độ Kích hoạt} \times \text{Độ dính Thói quen} \times \text{Sức bền Đường dài}$$